## Data Prepare ##

In [1]:
%cd /Users/ostrichzhang/Desktop/HKUST/25-26Spring/6000C/MFIT6000CAlgoTrading

/Users/ostrichzhang/Desktop/HKUST/25-26Spring/6000C/MFIT6000CAlgoTrading


/Users/ostrichzhang/WorkSpace/SoftWare/miniconda3/envs/ophr/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [11]:
import pandas as pd
import numpy as np

hsi_df=pd.read_csv('data/HSI_data.csv')
vhsi_df=pd.read_csv('data/VHSI_data.csv')
chain_df = pd.read_csv("data/Selected_HSI_Options.csv")


# 标准化日期
for df in (hsi_df, vhsi_df, chain_df):
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
chain_df["ExpDate"] = pd.to_datetime(chain_df["ExpDate"]).dt.normalize()

start = pd.Timestamp("2016-01-01")
end   = pd.Timestamp("2025-12-31")
hsi  = hsi_df[(hsi_df["Date"]>=start) & (hsi_df["Date"]<=end)].copy().reset_index(drop=True)
vhsi = vhsi_df[(vhsi_df["Date"]>=start) & (vhsi_df["Date"]<=end)].copy().reset_index(drop=True)
chain = chain_df[(chain_df["Date"]>=start) & (chain_df["Date"]<=end)].copy().reset_index(drop=True)

In [12]:
def parse_number(x):
    """Parse numbers like '21,123.0' -> 21123.0"""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace('"','').replace(',','')
    if s == "" or s.lower() in {"nan", "null"}:
        return np.nan
    try:
        return float(s)
    except:
        return np.nan
    


future_df = pd.read_csv('data/Hang Seng Futures Historical Price Data.csv')
future_df.columns = future_df.columns.str.replace('"', '', regex=False).str.strip()
future_df["Date"] = pd.to_datetime(future_df["Date"], dayfirst=True).dt.normalize()

for col in ["Price","Open","High","Low"]:
    future_df[col] = future_df[col].map(parse_number)


# 最终你大概率只需要 Date + F(=Price)
future_df = future_df.rename(columns={"Price":"Close"})[["Date","Close","Open","High","Low"]]
future_df = future_df.sort_values("Date").reset_index(drop=True)

future=future_df[(future_df["Date"]>=start) & (future_df["Date"]<=end)].copy().reset_index(drop=True)
future.head()

,Date,Close,Open,High,Low
0,2016-01-04,21123.0,21807.0,21857.0,21114.0
1,2016-01-05,21140.0,21138.0,21409.0,21020.0
2,2016-01-06,20861.0,21137.0,21250.0,20855.0
3,2016-01-07,20349.0,20852.0,20878.0,20258.0
4,2016-01-08,20399.0,20328.0,20620.0,20050.0


In [13]:
mkt=(hsi[["Date","Close"]].rename(columns={"Close": "S"})).merge((vhsi[["Date","Close"]].rename(columns={"Close": "IV_pct"})), on="Date", how="inner").reset_index(drop=True)
mkt=mkt.merge(future[["Date","Close"]].rename(columns={"Close": "F"}), on="Date", how="inner")
mkt["IV"] = mkt["IV_pct"] / 100
mkt.drop(columns=["IV_pct"], inplace=True)
merged_df = mkt.copy()

In [14]:
merged_df.head()

,Date,S,F,IV
0,2016-01-04,21327.119141,21123.0,0.2330
1,2016-01-05,21188.720703,21140.0,0.2325
2,2016-01-06,20980.810547,20861.0,0.2494
3,2016-01-07,20333.339844,20349.0,0.2887
4,2016-01-08,20453.710938,20399.0,0.2698


In [16]:
import pandas as pd
import numpy as np
import datetime as dt
from pandas.tseries.offsets import BDay

# ----------------------------
# Config
# ----------------------------
N_EXPIRIES = 3
TYPE_LIST = ("Call", "Put")
K_AROUND_ATM = (-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5)   
DEFAULT_STEP = 100
EXPIRY_RULE = "index_options"


# ----------------------------
# Expiry helpers
# ----------------------------
def third_friday(year: int, month: int) -> pd.Timestamp:
    d = dt.date(year, month, 15)
    while d.weekday() != 4:
        d += dt.timedelta(days=1)
    return pd.Timestamp(d)

def month_end(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    next_month = (ts.replace(day=1) + pd.offsets.MonthBegin(1))
    return next_month - pd.Timedelta(days=1)

def second_last_business_day_of_month(ts: pd.Timestamp) -> pd.Timestamp:
    me = month_end(ts)
    last_bd = me if me.weekday() < 5 else me - BDay(1)
    return last_bd - BDay(1)

def adjust_to_prev_business_day(x: pd.Timestamp) -> pd.Timestamp:
    x = pd.Timestamp(x)
    while x.weekday() >= 5:
        x -= pd.Timedelta(days=1)
    return x

def gen_expiries(date: pd.Timestamp, n: int = 3, rule: str = "futures_options") -> list[pd.Timestamp]:
    date = pd.Timestamp(date).normalize()
    y, m = date.year, date.month
    expiries = []

    i = 0
    while len(expiries) < n:
        mm = (m + i - 1) % 12 + 1
        yy = y + (m + i - 1) // 12

        if rule == "futures_options":
            exp = third_friday(yy, mm)
            exp = adjust_to_prev_business_day(exp)
        elif rule == "index_options":
            exp = second_last_business_day_of_month(pd.Timestamp(dt.date(yy, mm, 1)))
        else:
            raise ValueError("rule must be 'futures_options' or 'index_options'.")

        if exp > date:
            expiries.append(exp)

        i += 1

    return expiries


# ----------------------------
# Strike helpers
# ----------------------------
def strike_step_from_S(S: float, default_step: int = DEFAULT_STEP) -> int:
    if S >= 20000:
        return 200
    elif S >= 5000:
        return 100
    else:
        return 50

def atm_strike(S: float, step: int) -> int:
    return int(round(S / step) * step)


# ----------------------------
# Main: monthly-anchored strike grid
# ----------------------------
def build_synth_chain_monthly_anchor(
    df: pd.DataFrame,
    n_expiries: int = N_EXPIRIES,
    expiry_rule: str = EXPIRY_RULE,
    use_dynamic_step: bool = True,
    fixed_step: int = DEFAULT_STEP,
    anchor_col: str = "S",
) -> pd.DataFrame:
    """
    Build synthetic option chain where strikes are fixed within each calendar month,
    based on the monthly average underlying.

    Input df columns:
        - Date
        - S (or anchor_col)

    Output columns:
        - Date, Expiry, Strike, Type
        - month_anchor_S, strike_step, atm_anchor_strike
    """
    dff = df.copy()
    dff["Date"] = pd.to_datetime(dff["Date"]).dt.normalize()

    if anchor_col not in dff.columns:
        raise ValueError(f"Missing anchor column: {anchor_col}")

    # calendar month bucket
    dff["month_bucket"] = dff["Date"].dt.to_period("M")

    # monthly average underlying
    monthly_anchor = (
        dff.groupby("month_bucket", as_index=False)[anchor_col]
        .mean()
        .rename(columns={anchor_col: "month_anchor_S"})
    )

    dff = dff.merge(monthly_anchor, on="month_bucket", how="left")

    # monthly strike step and ATM anchor strike
    if use_dynamic_step:
        dff["strike_step"] = dff["month_anchor_S"].apply(strike_step_from_S)
    else:
        dff["strike_step"] = int(fixed_step)

    dff["atm_anchor_strike"] = [
        atm_strike(s, step)
        for s, step in zip(dff["month_anchor_S"], dff["strike_step"])
    ]

    rows = []
    for r in dff.itertuples(index=False):
        d = getattr(r, "Date")
        month_anchor_S = float(getattr(r, "month_anchor_S"))
        step = int(getattr(r, "strike_step"))
        atm = int(getattr(r, "atm_anchor_strike"))

        strikes = [atm + k * step for k in K_AROUND_ATM]
        expiries = gen_expiries(d, n=n_expiries, rule=expiry_rule)

        for exp in expiries:
            for K in strikes:
                for t in TYPE_LIST:
                    rows.append((
                        d,
                        exp,
                        int(K),
                        t,
                        month_anchor_S,
                        step,
                        atm
                    ))

    out = pd.DataFrame(
        rows,
        columns=[
            "Date", "Expiry", "Strike", "Type",
            "month_anchor_S", "strike_step", "atm_anchor_strike"
        ]
    )
    return out


# ----------------------------
# Example usage
# ----------------------------

chain = build_synth_chain_monthly_anchor(merged_df, n_expiries=3, expiry_rule="index_options", use_dynamic_step=True, anchor_col="S")
sel=chain.merge(merged_df, on="Date", how="left")
sel["dte"] = (sel["Expiry"] - sel["Date"]).dt.days
sel.head(20)

,Date,Expiry,Strike,Type,month_anchor_S,strike_step,atm_anchor_strike,S,F,IV,dte
0,2016-01-04,2016-01-28,19200,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24
1,2016-01-04,2016-01-28,19200,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24
2,2016-01-04,2016-01-28,19300,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24
3,2016-01-04,2016-01-28,19300,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24
4,2016-01-04,2016-01-28,19400,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24
5,2016-01-04,2016-01-28,19400,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24
6,2016-01-04,2016-01-28,19500,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24
7,2016-01-04,2016-01-28,19500,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24
8,2016-01-04,2016-01-28,19600,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24
9,2016-01-04,2016-01-28,19600,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24


In [ ]:


# # ---- 选择 2 nearest expiry + ATM±2 strikes ----
# out = []
# for d, g in chain_call.groupby("Date"):
#     # 只考虑到期在 d 之后的
#     expiries = np.sort(g.loc[g["ExpDate"] > d, "ExpDate"].unique())
#     if len(expiries) == 0:
#         continue
#     expiries = expiries[:4]  # 2 nearest expiry

#     S = float(g["S"].iloc[0])

#     for exp in expiries:
#         gg = g[g["ExpDate"] == exp].copy()
#         strikes = np.unique(np.sort(gg["StrikePrice"].astype(int).values))
#         if len(strikes) == 0:
#             continue

#         # ATM = 最近 strike
#         idx_atm = int(np.argmin(np.abs(strikes - S)))
#         lo = max(idx_atm - 2, 0)
#         hi = min(idx_atm + 2, len(strikes) - 1)
#         target = set(strikes[lo:hi+1])

#         gg = gg[gg["StrikePrice"].astype(int).isin(target)].copy()
#         gg["atm_strike"] = int(strikes[idx_atm])
#         gg["dte"] = (gg["ExpDate"] - gg["Date"]).dt.days

#         out.append(gg)

# sel = pd.concat(out, ignore_index=True)


In [17]:
import math
# ========= 2) Black-Scholes Call + Delta =========
def norm_cdf(x):
    return 0.5*(1.0 + math.erf(x/math.sqrt(2.0)))

def bs_call_delta(S, K, T, r, q, sigma):
    # 返回：call_price (index points), delta
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return np.nan, np.nan
    vol_sqrt = sigma * math.sqrt(T)
    d1 = (math.log(S/K) + (r - q + 0.5*sigma*sigma)*T) / vol_sqrt
    d2 = d1 - vol_sqrt
    call = S*math.exp(-q*T)*norm_cdf(d1) - K*math.exp(-r*T)*norm_cdf(d2)
    delta = math.exp(-q*T)*norm_cdf(d1)
    return call, delta

def black_call_delta(F, K, T, r, sigma):
    # 返回：call_price (index points), delta
    if T <= 0 or sigma <= 0 or F <= 0 or K <= 0:
        return np.nan, np.nan
    vol_sqrt = sigma * math.sqrt(T)
    d1 = (math.log(F/K) + (0.5*sigma*sigma)*T) / vol_sqrt
    d2 = d1 - vol_sqrt
    call = math.exp(-r*T)*(F*norm_cdf(d1) - K*norm_cdf(d2))
    delta = math.exp(-r*T)*norm_cdf(d1)
    return call, delta

def black_put_delta(F, K, T, r, sigma):
    # 返回：put_price (same units as F,K), delta wrt F
    if F <= 0 or K <= 0:
        return np.nan, np.nan

    disc = math.exp(-r * max(T, 0.0))

    # expiry
    if T <= 0:
        price = max(K - F, 0.0)
        delta = -1.0 if F < K else 0.0
        return price, delta

    # zero vol
    if sigma <= 0:
        price = disc * max(K - F, 0.0)
        delta = disc * (-1.0 if F < K else 0.0)
        return price, delta

    vol_sqrt = sigma * math.sqrt(T)
    d1 = (math.log(F / K) + 0.5 * sigma * sigma * T) / vol_sqrt
    d2 = d1 - vol_sqrt

    put = disc * (K * norm_cdf(-d2) - F * norm_cdf(-d1))
    delta = -disc * norm_cdf(-d1)  # = disc*(norm_cdf(d1)-1)
    return put, delta

def sigma_from_vhsi(vhsi, dte):
    sigma30 = float(vhsi)
    dte = max(int(dte), 1)
    return sigma30 * math.sqrt(30.0 / dte)

In [18]:
backup = []
r = 0.02  # Black-76里不需要 q；q 已经体现在 F 里

for _, row in sel.iterrows():
    F = float(row["F"])
    K = float(row["Strike"])
    dte = int(row["dte"])
    T = dte / 365.0

    # IV：建议统一成小数
    iv_raw = float(row["IV"])
    sigma = iv_raw / 100.0 if iv_raw > 2 else iv_raw  # 粗略判定：>2 就当百分比

    opt_type = str(row["Type"]).strip().lower()

    if opt_type == "call":
        price, delta = black_call_delta(F, K, T, r, sigma)
    elif opt_type == "put":
        price, delta = black_put_delta(F, K, T, r, sigma)
    else:
        price, delta = np.nan, np.nan

    # 你可以把 source 写成 "chain_iv" 或 "vhsi" 看你 sigma 来源
    source = "chain_iv"  # 这里你用的是 row["IV"]，所以更像 chain_iv

    backup.append((sigma, price, delta))

sel["sigma_used"] = [x[0] for x in backup]
sel["price_fut"] = [x[1] for x in backup]
sel["delta_fut"] = [x[2] for x in backup]

In [19]:
parent_path='data/'
sel.to_csv(f"{parent_path}chain_with_black_prices.csv", index=False)
sel.head(10)

,Date,Expiry,Strike,Type,month_anchor_S,strike_step,atm_anchor_strike,S,F,IV,dte,sigma_used,price_fut,delta_fut
0,2016-01-04,2016-01-28,19200,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,1948.549623,0.946937
1,2016-01-04,2016-01-28,19200,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,28.076838,-0.051749
2,2016-01-04,2016-01-28,19300,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,1855.051269,0.937052
3,2016-01-04,2016-01-28,19300,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,34.447063,-0.061634
4,2016-01-04,2016-01-28,19400,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,1762.707892,0.925814
5,2016-01-04,2016-01-28,19400,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,41.972266,-0.072872
6,2016-01-04,2016-01-28,19500,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,1671.665480,0.913140
7,2016-01-04,2016-01-28,19500,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,50.798433,-0.085546
8,2016-01-04,2016-01-28,19600,Call,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,1582.076761,0.898961
9,2016-01-04,2016-01-28,19600,Put,19733.578613,100,19700,21327.119141,21123.0,0.233,24,0.233,61.078294,-0.099725


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
import matplotlib.pyplot as plt


d = pd.Timestamp("2024-01-02")   # 你想画哪一天就填哪天
g = sel[sel["Date"] == d].copy()
# 如果没有 dte 列就加一列
if "dte" not in g.columns:
    g["dte"] = (g["ExpDate"] - g["Date"]).dt.days

# 真实价格矩阵：行=Strike，列=DTE
Z_real = g.pivot_table(index="StrikePrice", columns="dte", values="SettlementPrice", aggfunc="mean")

# backup 价格矩阵
Z_fut  = g.pivot_table(index="StrikePrice", columns="dte", values="price_fut", aggfunc="mean")
Z_S= g.pivot_table(index="StrikePrice", columns="dte", values="price_S", aggfunc="mean")

def plot_surface(Z, title):
    # Z: DataFrame index=strike, columns=dte
    strikes = Z.index.values.astype(float)
    dtes = Z.columns.values.astype(float)
    X, Y = np.meshgrid(dtes, strikes)     # X=dte, Y=strike
    Zval = Z.values.astype(float)

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(X, Y, Zval)
    ax.set_xlabel("DTE")
    ax.set_ylabel("Strike")
    ax.set_zlabel("Call Price (pts)")
    ax.set_title(title)
    plt.show()

Z_diff_fut = Z_fut - Z_real
Z_diff_S = Z_S - Z_real



def plot_heatmap(Z, title):
    # 统一排序
    Z = Z.sort_index().sort_index(axis=1)
    fig = plt.figure()
    ax = fig.add_subplot(111)
    im = ax.imshow(Z.values, aspect="auto", origin="lower")
    ax.set_title(title)
    ax.set_xlabel("DTE")
    ax.set_ylabel("Strike")

    # 设置刻度标签
    ax.set_xticks(np.arange(len(Z.columns)))
    ax.set_xticklabels(Z.columns.tolist())
    ax.set_yticks(np.arange(len(Z.index)))
    ax.set_yticklabels(Z.index.tolist())

    fig.colorbar(im, ax=ax)
    plt.show()

plot_surface(Z_real, f"Real Settlement Price Surface (Call) — {d.date()}")
plot_surface(Z_fut,  f"Future Price Surface (Call) — {d.date()}")
plot_surface(Z_S, f"Stock Price Surface (Call) — {d.date()}")
plot_surface(Z_diff_fut, f"Future - Real Price Surface (Call) — {d.date()}")
plot_surface(Z_diff_S, f"Stock - Real Price Surface (Call) — {d.date()}")

# plot_heatmap(Z_real, f"Real Settlement Heatmap (Call) — {d.date()}")
# plot_heatmap(Z_fut,  f"Future Heatmap (Call) — {d.date()}")
# plot_heatmap(Z_S, f"Stock Heatmap (Call) — {d.date()}")
plot_heatmap(Z_diff_fut, f"Future - Real Heatmap (Call) — {d.date()}")
plot_heatmap(Z_diff_S, f"Stock - Real Heatmap (Call) — {d.date()}")


In [102]:


df = sel.copy()  # 或 pd.read_csv("chain_with_black_prices.csv", parse_dates=["Date","Expiry"])

PRICE = "price_fut"
KCOL  = "Strike"

def check_rationality(df: pd.DataFrame) -> dict:
    d = df.copy()
    d["Date"] = pd.to_datetime(d["Date"])
    d["Expiry"] = pd.to_datetime(d["Expiry"])

    # ---------- A) Parity ----------
    wide = (
        d.pivot_table(index=["Date","Expiry",KCOL], columns="Type", values=PRICE, aggfunc="first")
         .reset_index()
    )
    # bring F from original (should be constant within Date/Expiry, but we join safely)
    F_map = d.groupby(["Date","Expiry"])[["F"]].first().reset_index()
    wide = wide.merge(F_map, on=["Date","Expiry"], how="left")

    wide["parity_rhs"] = wide["F"] - wide[KCOL]
    wide["parity_lhs"] = wide["Call"] - wide["Put"]
    wide["parity_err"] = wide["parity_lhs"] - wide["parity_rhs"]
    parity_summary = wide["parity_err"].abs().describe()

    # ---------- B) Bounds ----------
    dd = d.merge(F_map, on=["Date","Expiry"], how="left", suffixes=("","_g"))
    Fv = dd["F_g"].fillna(dd["F"])
    K  = dd[KCOL].astype(float)
    P  = dd[PRICE].astype(float)

    lower = np.where(dd["Type"].eq("Call"), np.maximum(Fv - K, 0.0), np.maximum(K - Fv, 0.0))
    upper = np.where(dd["Type"].eq("Call"), Fv, K)

    dd["bound_violation"] = (P < lower - 1e-8) | (P > upper + 1e-8)
    bounds_cnt = int(dd["bound_violation"].sum())

    # ---------- C) Monotonicity & D) Convexity ----------
    mono_viol = 0
    convex_viol = 0

    for (date, exp, opt_type), g in d.groupby(["Date","Expiry","Type"]):
        g = g.sort_values(KCOL)
        prices = g[PRICE].to_numpy()
        strikes = g[KCOL].to_numpy()

        # monotonic
        diff = np.diff(prices)
        if opt_type == "Call":
            mono_viol += int((diff > 1e-8).sum())  # should be <=0
        else:
            mono_viol += int((diff < -1e-8).sum()) # should be >=0

        # convexity (need at least 3 points)
        if len(prices) >= 3:
            # assumes roughly equal strike spacing; still works as a quick check here
            second = prices[:-2] - 2*prices[1:-1] + prices[2:]
            convex_viol += int((second < -1e-8).sum())

    # ---------- return ----------
    return {
        "parity_err_abs_stats": parity_summary,
        "bounds_violations_rows": bounds_cnt,
        "monotonicity_violations_count": mono_viol,
        "convexity_violations_count": convex_viol,
        "worst_parity_rows": wide.loc[wide["parity_err"].abs().nlargest(10).index,
                                      ["Date","Expiry",KCOL,"F","Call","Put","parity_err"]]
    }

res = check_rationality(df)
print(res["parity_err_abs_stats"])
print("bounds_violations_rows:", res["bounds_violations_rows"])
print("monotonicity_violations_count:", res["monotonicity_violations_count"])
print("convexity_violations_count:", res["convexity_violations_count"])
print(res["worst_parity_rows"])

count    25830.000000
mean         0.572611
std          0.575977
min          0.000000
25%          0.140025
50%          0.389632
75%          0.827091
max          7.214061
Name: parity_err, dtype: float64
bounds_violations_rows: 0
monotonicity_violations_count: 0
convexity_violations_count: 0
            Date     Expiry  Strike        F         Call          Put  \
25004 2025-10-10 2025-12-30   26600  24971.0   449.536388  2071.322327   
7604  2021-01-20 2021-03-30   30400  28700.0   451.212743  2144.797481   
25003 2025-10-10 2025-12-30   26400  24971.0   502.814039  1925.485682   
21295 2024-10-09 2024-12-30   20200  21588.0  2341.912955   960.135458   
23114 2025-04-08 2025-06-27   20600  19303.0   894.156658  2185.483621   
7603  2021-01-20 2021-03-30   30200  28700.0   504.050086  1998.389561   
25002 2025-10-10 2025-12-30   26200  24971.0   560.940108  1784.497455   
21296 2024-10-09 2024-12-30   20400  21588.0  2223.942775  1041.268664   
7602  2021-01-20 2021-03-30   30000 